In [ ]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_ALLOC_CONF']     = 'expandable_segments:True'

!pip uninstall -y unsloth transformers peft trl accelerate xformers bitsandbytes 2>/dev/null
!pip install --no-cache-dir \
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git' \
    pyyaml datasets evaluate rouge_score huggingface_hub -q

print('✅ Dependencies installed')

In [ ]:
import pathlib

for fname in ['config.yaml', 'med_dataset.json']:
    status = '✅' if pathlib.Path(fname).exists() else '❌  MISSING — upload before running Cell 3'
    print(f'{status}  {fname}')

In [ ]:
import yaml, json, random, pathlib, os
import torch
from datasets            import load_dataset, Dataset, DatasetDict
from unsloth             import FastLanguageModel
from trl                 import SFTTrainer
from transformers        import TrainingArguments
from huggingface_hub     import login
from google.colab        import userdata


# ── Config ────────────────────────────────────────────────────
def load_config(path='config.yaml'):
    p = pathlib.Path(path)
    if not p.exists():
        raise FileNotFoundError(f"'{path}' not found. Upload it first.")
    cfg = yaml.safe_load(p.read_text())
    for s in ['model','lora','dataset','training','inference','export']:
        if s not in cfg: raise KeyError(f'config.yaml missing section: {s}')
    return cfg


# ── Local dataset ─────────────────────────────────────────────
def load_local_dataset(dataset_cfg):
    p = pathlib.Path(dataset_cfg['local_path'])
    if not p.exists():
        raise FileNotFoundError(f"'{p}' not found. Upload it first.")
    data = json.loads(p.read_text())
    print(f'  Local data: {len(data)} samples')
    return data


# ── Row extractor ─────────────────────────────────────────────
def _extract_row(row, src_cfg):
    q       = str(row.get(src_cfg.get('input_col',  'input'),  '')).strip()
    raw_ans = row.get(src_cfg.get('output_col', 'output'))
    a       = str(raw_ans).strip() if raw_ans and str(raw_ans).lower() not in ('nan','none','') else ''
    if not a and src_cfg.get('fallback_context_col'):
        ctx = row.get(src_cfg['fallback_context_col'], {})
        key = src_cfg.get('fallback_context_key', '')
        if isinstance(ctx, dict) and key and key in ctx:
            a = ' '.join(str(p) for p in ctx[key] if p)
        elif isinstance(ctx, list):
            a = ' '.join(str(p) for p in ctx if p)
    if not a and src_cfg.get('correct_idx_col') and src_cfg.get('option_cols'):
        idx = row.get(src_cfg['correct_idx_col'])
        if idx is not None:
            try:   a = str(row.get(src_cfg['option_cols'][int(idx)], '')).strip()
            except: pass
    return q, a


# ── Source loader ─────────────────────────────────────────────
def load_source(src, variants, local_rows):
    limit = src.get('max_samples', 9999)
    print(f'  Loading {src["name"]} ...')
    if src['type'] == 'local':
        rows = local_rows
    elif src['type'] == 'huggingface':
        kwargs = {'split': src.get('split', 'train')}
        if src.get('config'): kwargs['name'] = src['config']
        ds   = load_dataset(src['path'], **kwargs)
        rows = [dict(r) for r in ds.select(range(min(limit, len(ds))))]
    else:
        raise ValueError(f'Unknown source type: {src["type"]}')
    normalised = []
    for row in rows:
        q, a = _extract_row(row, src)
        if q and a:
            normalised.append({'instruction': random.choice(variants), 'input': q, 'output': a})
    print(f'    → {len(normalised)} usable samples')
    return normalised


# ── Dataset builder ───────────────────────────────────────────
def build_dataset(dataset_cfg, local_rows):
    random.seed(dataset_cfg.get('seed', 42))
    all_rows = []
    for src in dataset_cfg['sources']:
        if not src.get('enabled', True):
            print(f'  Skipping {src["name"]}')
            continue
        try:
            all_rows.extend(load_source(src, dataset_cfg['instruction_variants'], local_rows))
        except Exception as e:
            print(f'  WARNING: {src["name"]} failed — {e}')
    random.shuffle(all_rows)
    cap  = dataset_cfg.get('total_cap', 5000)
    cut  = int(len(all_rows[:cap]) * (1 - dataset_cfg.get('val_split', 0.05)))
    rows = all_rows[:cap]
    print(f'  Dataset: {len(rows)} total → train {cut} / val {len(rows)-cut}')
    return DatasetDict({'train': Dataset.from_list(rows[:cut]), 'validation': Dataset.from_list(rows[cut:])})


# ── Model loader ──────────────────────────────────────────────
def load_model(model_cfg, lora_cfg):
    model, tokenizer = FastLanguageModel.from_pretrained(**model_cfg)
    model = FastLanguageModel.get_peft_model(model, **lora_cfg)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'  Trainable params: {trainable:,} ({100*trainable/total:.2f}%)')
    return model, tokenizer


# ── Prompt formatter ──────────────────────────────────────────
def prepare_splits(splits, tokenizer, model_cfg):
    eos     = tokenizer.eos_token
    max_len = model_cfg['max_seq_length']
    def fmt(ex):
        return {'text': f"### Instruction:\n{ex['instruction']}\n\n### Input:\n{ex['input']}\n\n### Response:\n{ex['output']}{eos}"}
    def fits(ex):
        return len(tokenizer(ex['text'], truncation=False)['input_ids']) <= max_len
    cols = ['instruction','input','output']
    train_ds = splits['train'].map(fmt, remove_columns=cols).filter(fits, num_proc=1)
    val_ds   = splits['validation'].map(fmt, remove_columns=cols).filter(fits, num_proc=1)
    print(f'  After filter — train: {len(train_ds)} / val: {len(val_ds)}')
    return train_ds, val_ds


# ── Export ────────────────────────────────────────────────────
def export_model(model, tokenizer, export_cfg):
    token = userdata.get(export_cfg.get('hub_token_env', 'HF_TOKEN'))
    os.environ['HF_TOKEN'] = token
    login(token=token)
    hub_id = export_cfg['hub_model_id']
    model.push_to_hub(hub_id)
    tokenizer.push_to_hub(hub_id)
    print(f'  Pushed → https://huggingface.co/{hub_id}')


# ══════════════════════════════════════════════════════════════
#  MAIN  — entire pipeline in one call
# ══════════════════════════════════════════════════════════════
def main(config_path='config.yaml'):
    print('\n━━━ 1 / 6  Loading config')
    cfg          = load_config(config_path)
    model_cfg    = cfg['model']
    lora_cfg     = cfg['lora']
    dataset_cfg  = cfg['dataset']
    training_cfg = cfg['training']
    export_cfg   = cfg['export']

    print('\n━━━ 2 / 6  Loading local dataset')
    local_data = load_local_dataset(dataset_cfg)

    print('\n━━━ 3 / 6  Building dataset')
    splits = build_dataset(dataset_cfg, local_data)

    print('\n━━━ 4 / 6  Loading model + LoRA')
    model, tokenizer = load_model(model_cfg, lora_cfg)

    print('\n━━━ 5 / 6  Formatting prompts + training')
    train_ds, val_ds = prepare_splits(splits, tokenizer, model_cfg)
    torch._dynamo.config.disable = True
    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer,
        train_dataset=train_ds, eval_dataset=val_ds,
        dataset_text_field='text',
        max_seq_length=model_cfg['max_seq_length'],
        packing=False,
        args=TrainingArguments(**training_cfg),
    )
    result = trainer.train()
    m = result.metrics
    print(f'  Runtime: {m["train_runtime"]:.0f}s  |  Loss: {m["train_loss"]:.4f}')

    print('\n━━━ 6 / 6  Pushing to HuggingFace Hub')
    export_model(model, tokenizer, export_cfg)

    print('\n✅  Training complete. Use demo.ipynb to present results.')
    return model, tokenizer, cfg


# ── Run ───────────────────────────────────────────────────────
model, tokenizer, cfg = main()